# Search and A*
This notebook is the solution for the lab `Search.pdf` (Warehouse Robot Navigation with A*).

**LLM use:** the A*/BFS code below was generated with Claude (Anthropic) from the prompt in Task 2 and then reviewed, modified and tested. Changes made to the generated code are listed in Task 7.

### Task 0: Understand the search problem

| Component | Specification |
|---|---|
| State $S$ | The robot's cell $(r, c)$ on the grid, restricted to non-`#` cells |
| Actions $A$ | `Up`, `Down`, `Left`, `Right` |
| Transition $T$ | $T((r,c), a) = (r + \Delta r_a,\ c + \Delta c_a)$ if that cell is inside the grid and not `#`; otherwise the action is not applicable |
| Initial state $s_0$ | The cell marked `S`, i.e. $(1, 1)$ |
| Goal $G$ | The cell marked `G`, i.e. $(7, 15)$ |
| Cost $c$ | $c(s, a, s') = 1$ for every move |

(a) **What is needed to specify a state?** Only the robot's position $(r, c)$. The map is static, so it is part of the environment rather than the state.

(b) **What makes an action invalid?** If the target cell is a wall `#` or lies outside the grid.

(c) **Is it deterministic?** Yes. Each action from a given cell has exactly one outcome, the map never changes, and the environment is fully observable.

(d) **What is a solution?** A sequence of actions (equivalently, a sequence of adjacent free cells) from `S` to `G`. An *optimal* solution is one with minimum total cost, i.e. the fewest moves.

### Task 1: Plan the agent (design written before prompting the LLM)

1. **State representation:** a tuple `(row, col)` (hashable, so it can go into sets and dicts).
2. **Warehouse representation:** a list of lists of characters parsed from the ASCII map; `S` and `G` are located during parsing.
3. **Valid actions:** for each of the four moves, compute the neighbour and keep it only if it is in bounds and not `#`.
4. **Goal test:** `state == goal`, performed when a node is **popped** from the frontier (not when it is generated). For A* this is what guarantees optimality.
5. **Frontier:** a min-priority queue (`heapq`) of entries `(f, h, counter, g, state)`. `f = g + h` is the priority; `h` breaks ties in favour of nodes closer to the goal; `counter` keeps ordering stable and avoids comparing states.
6. **Path reconstruction:** a `parent` dict mapping each state to the state it was reached from; walk back from the goal to `S` and reverse.

Also kept: `g_cost` (best known cost to each state) and a `closed` set of already-expanded states.

**Reported on termination:** solution found (yes/no), the path, the path length, and the number of states expanded.

### Task 2: Ask an LLM to generate A*

**Prompt used:**

> I am implementing a simple goal-based search agent in Python.
> The environment is a grid represented by an ASCII map. The agent starts at S and must reach G. The symbols # represent obstacles and . represents free cells. The agent can move up, down, left, or right, and every movement has cost 1.
> Implement A* search. Use Manhattan distance as the heuristic: h(n) = |x − x_G| + |y − y_G|.
> The program should: represent grid positions as states; maintain an appropriate frontier; calculate g(n), h(n) and f(n); avoid repeatedly expanding the same state; reconstruct the path when the goal is reached; report the path and its length; report the number of states expanded.
> Make the heuristic a parameter so it can be swapped, and also give me a BFS version with the same return format so the two can be compared.
> Keep the implementation simple and explain the main components of the code.

The final (reviewed and modified) code is below.

In [1]:
import heapq
import itertools
import math
from collections import deque

# Actions: name -> (d_row, d_col)
MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}


def parse_map(text):
    """Parse an ASCII map into a grid plus the start and goal cells."""
    grid = [list(row) for row in text.strip("\n").splitlines()]
    start = goal = None
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
    return grid, start, goal


def successors(grid, state):
    """Transition model: yield (action, next_state) for every applicable action."""
    r, c = state
    for action, (dr, dc) in MOVES.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[nr]) and grid[nr][nc] != "#":
            yield action, (nr, nc)


# ---- heuristics -------------------------------------------------------------
def manhattan(s, g):
    return abs(s[0] - g[0]) + abs(s[1] - g[1])

def zero(s, g):
    return 0

def euclidean(s, g):
    return math.hypot(s[0] - g[0], s[1] - g[1])

def double_manhattan(s, g):
    return 2 * manhattan(s, g)


def reconstruct(parent, state):
    path = [state]
    while parent[state] is not None:
        state = parent[state]
        path.append(state)
    return path[::-1]


def astar(grid, start, goal, h=manhattan):
    counter = itertools.count()
    h0 = h(start, goal)
    frontier = [(h0, h0, next(counter), 0, start)]   # (f, h, tie-break, g, state)
    g_cost = {start: 0}
    parent = {start: None}
    closed = set()
    expanded = 0

    while frontier:
        f, _, _, g, state = heapq.heappop(frontier)
        if state in closed:            # stale entry: a cheaper copy was already expanded
            continue
        if state == goal:              # goal test on expansion -> optimal with admissible h
            path = reconstruct(parent, state)
            return {"found": True, "path": path, "length": len(path) - 1, "expanded": expanded}
        closed.add(state)
        expanded += 1
        for _, nxt in successors(grid, state):
            new_g = g + 1              # step cost is always 1
            if nxt not in closed and new_g < g_cost.get(nxt, float("inf")):
                g_cost[nxt] = new_g
                parent[nxt] = state
                hn = h(nxt, goal)
                heapq.heappush(frontier, (new_g + hn, hn, next(counter), new_g, nxt))   # f = g + h

    return {"found": False, "path": None, "length": None, "expanded": expanded}


def bfs(grid, start, goal):
    frontier = deque([start])
    parent = {start: None}             # doubles as the "reached" set
    expanded = 0
    while frontier:
        state = frontier.popleft()
        if state == goal:
            path = reconstruct(parent, state)
            return {"found": True, "path": path, "length": len(path) - 1, "expanded": expanded}
        expanded += 1
        for _, nxt in successors(grid, state):
            if nxt not in parent:
                parent[nxt] = state
                frontier.append(nxt)
    return {"found": False, "path": None, "length": None, "expanded": expanded}


def draw(grid, path):
    out = [row[:] for row in grid]
    for r, c in path or []:
        if out[r][c] == ".":
            out[r][c] = "*"
    return "\n".join("".join(row) for row in out)


def report(name, res, grid=None):
    print(f"[{name}] found={res['found']}  length={res['length']}  expanded={res['expanded']}")
    if res["found"] and grid is not None:
        print(draw(grid, res["path"]))

### Task 3: Test the generated program

Each test states its expected result as an `assert`, so the notebook fails loudly if the code is wrong.

**Test 1: Original warehouse**

In [2]:
WAREHOUSE = """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
"""
grid, start, goal = parse_map(WAREHOUSE)
res = astar(grid, start, goal)
report("A* / warehouse", res, grid)
print("path:", res["path"])

# Independent validity check of the returned path
p = res["path"]
assert p[0] == start and p[-1] == goal
assert all(grid[r][c] != "#" for r, c in p)
assert all(manhattan(a, b) == 1 for a, b in zip(p, p[1:])), "every step must be one orthogonal move"
assert res["length"] == bfs(grid, start, goal)["length"], "A* length must match BFS (optimal)"
print("Test 1 passed")

[A* / warehouse] found=True  length=40  expanded=63
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################
path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
Test 1 passed


**Test 2: Trivial case** (goal adjacent to start)

In [3]:
grid2, s2, g2 = parse_map("""
#####
#SG##
#####
""")
res2 = astar(grid2, s2, g2)
report("A* / trivial", res2, grid2)
assert res2["found"] and res2["length"] == 1 and res2["path"] == [s2, g2]
print("Test 2 passed")

[A* / trivial] found=True  length=1  expanded=1
#####
#SG##
#####
Test 2 passed


**Test 3: No solution** (goal walled off)

In [4]:
grid3, s3, g3 = parse_map("""
#######
#S....#
###.###
#...#G#
#######
""")
res3 = astar(grid3, s3, g3)
report("A* / no solution", res3)
assert not res3["found"] and res3["path"] is None
print("Test 3 passed: terminated with failure after exhausting the reachable states")

[A* / no solution] found=False  length=None  expanded=9
Test 3 passed: terminated with failure after exhausting the reachable states


**Test 4: Alternative paths**

Two different routes from `S` to `G`, both of length 14: along the top and down the right edge, or down the left side and through the middle/bottom. Other non-shortest routes also exist via the inner loop. The returned path must be *a* shortest path, so we compare its length against BFS, which is optimal for unit costs.

In [5]:
ALT = """
###########
#S........#
#.#######.#
#.#.....#.#
#.#.###.#.#
#.......#.#
#####.###.#
#........G#
###########
"""
grid4, s4, g4 = parse_map(ALT)
res4 = astar(grid4, s4, g4)
report("A* / alternative paths", res4, grid4)
assert res4["length"] == bfs(grid4, s4, g4)["length"]
print("Test 4 passed: A* returned a shortest path")

[A* / alternative paths] found=True  length=14  expanded=14
###########
#S........#
#*#######.#
#*#.....#.#
#*#.###.#.#
#*****..#.#
#####*###.#
#....****G#
###########
Test 4 passed: A* returned a shortest path


**Test results**

| Test | Expected | Found? | Length | Expanded | Correct? |
|---|---|---|---:|---:|---|
| 1. Original warehouse | path exists | Yes | 40 | 63 | Yes (valid, matches BFS) |
| 2. Trivial | 1 step | Yes | 1 | 1 | Yes |
| 3. No solution | failure | No | – | 9 | Yes (terminates after the 9 reachable cells) |
| 4. Alternative paths | shortest path | Yes | 14 | 14 | Yes (matches BFS) |

### Task 4: Inspect the A* algorithm

| Concept | Where it appears in the code |
|---|---|
| State | `(row, col)` tuples, e.g. `start`, `state`, `nxt` |
| Action | keys of `MOVES` (`Up/Down/Left/Right`) |
| Transition | `successors()` – applies the move and filters walls / out-of-bounds |
| Goal test | `if state == goal:` right after popping from the heap |
| $g(n)$ | `g` stored in the heap entry; `new_g = g + 1`; best values in `g_cost` |
| $h(n)$ | `hn = h(nxt, goal)` (the `h` argument, default `manhattan`) |
| $f(n)$ | `new_g + hn`, the first element of the heap tuple |
| Frontier | `frontier`, a binary heap managed with `heapq` |
| Visited states | `closed` (expanded) and `g_cost` (best cost seen so far) |
| Path reconstruction | `parent` dict + `reconstruct()` |

(a) **Frontier data structure:** a binary min-heap (`heapq`) of tuples `(f, h, counter, g, state)`.

(b) **How the next state is selected:** `heappop` returns the entry with the smallest `f`; ties go to the smaller `h` (closer to the goal), then to insertion order.

(c) **Where the heuristic is computed:** when a successor is generated (`hn = h(nxt, goal)`), and once for the start node.

(d) **Is $f = g + h$ explicit?** Yes, `new_g + hn` is computed and used as the heap priority.

(e) **Avoiding repeated exploration:** a state is expanded at most once (`closed`); a successor is only pushed if it gives a strictly better `g` than any seen before (`g_cost`); outdated heap entries are skipped when popped (lazy deletion).

### Task 5: Compare A* with blind search (BFS) on the same warehouse

In [6]:
grid, start, goal = parse_map(WAREHOUSE)
r_bfs, r_astar = bfs(grid, start, goal), astar(grid, start, goal)
print(f"{'Measure':<16}{'BFS':>8}{'A*':>8}")
print(f"{'Solution found':<16}{str(r_bfs['found']):>8}{str(r_astar['found']):>8}")
print(f"{'Path length':<16}{r_bfs['length']:>8}{r_astar['length']:>8}")
print(f"{'States expanded':<16}{r_bfs['expanded']:>8}{r_astar['expanded']:>8}")
free = sum(ch != "#" for row in grid for ch in row)
print("\nTotal free cells in the map:", free)

Measure              BFS      A*
Solution found      True    True
Path length           40      40
States expanded       63      63

Total free cells in the map: 64


| Measure | BFS | A* |
|---|---:|---:|
| Solution found | Yes | Yes |
| Path length | 40 | 40 |
| States expanded | 63 | 63 |

(a) Both found a solution.

(b) Both paths have length 40. Both are optimal here: BFS because every step costs 1, A* because Manhattan distance is admissible.

(c) **Neither** – both expanded 63 of the 64 free cells. This was not what I expected before running it.

(d) Why A* did not help here: there is only **one** corridor to `G`, and it goes *up* and to the right and then all the way down column 15. Manhattan distance measures straight-line grid distance and ignores walls, so it keeps rating the dead-end region in the bottom-left as "close" to `G` (it is in the bottom rows). A* is therefore pulled into every dead end before reaching the right-hand corridor, and ends up expanding almost everything – the same as BFS.

In general A* expands fewer states because it only expands nodes with $f(n) \le C^*$, and a good heuristic makes $f$ large for nodes that lead away from the goal. When the heuristic is misleading (walls force long detours), that pruning disappears. The maps in Task 6 show the opposite case, where A* is much better than BFS.

### Task 6: Investigate the heuristic

**Why Manhattan distance is appropriate (LLM explanation, checked):** with only 4-directional unit-cost moves, the robot needs at least $|\Delta r|$ vertical and $|\Delta c|$ horizontal moves even with no walls, so Manhattan distance never overestimates the true cost ($h \le h^*$, admissible). It is also consistent: one move changes $h$ by exactly 1 and costs 1. Walls can only make the real path longer.

Experiments: four heuristics, run on the lab warehouse and on three extra maps:
- `ALT` – two routes (from Test 4);
- `OPEN` – a large mostly open room with one internal wall;
- `TRAP` – a cluttered map where the most "direct-looking" route is not the shortest.

In [7]:
OPEN = """
####################
#S.................#
#..................#
#.......####.......#
#.......#..........#
#.......#..........#
#.......#..........#
#..................#
#.................G#
####################
"""
TRAP = """
#############
#S#.##......#
#..#..#.....#
#.......##..#
#....#..##..#
#..#....#..##
#...##..##..#
##.#......#G#
#############
"""
heuristics = [("h = 0", zero), ("Manhattan", manhattan), ("Euclidean", euclidean), ("2 x Manhattan", double_manhattan)]

for map_name, text in [("WAREHOUSE", WAREHOUSE), ("ALT", ALT), ("OPEN", OPEN), ("TRAP", TRAP)]:
    g_, s_, t_ = parse_map(text)
    b = bfs(g_, s_, t_)
    print(f"\n{map_name}: BFS (optimal) length = {b['length']}, BFS expanded = {b['expanded']}")
    print(f"  {'heuristic':<15}{'found':>7}{'length':>8}{'expanded':>10}{'optimal?':>10}")
    for name, h in heuristics:
        r = astar(g_, s_, t_, h)
        print(f"  {name:<15}{str(r['found']):>7}{r['length']:>8}{r['expanded']:>10}{str(r['length'] == b['length']):>10}")


WAREHOUSE: BFS (optimal) length = 40, BFS expanded = 63
  heuristic        found  length  expanded  optimal?
  h = 0             True      40        63      True
  Manhattan         True      40        63      True
  Euclidean         True      40        63      True
  2 x Manhattan     True      40        63      True

ALT: BFS (optimal) length = 14, BFS expanded = 40
  heuristic        found  length  expanded  optimal?
  h = 0             True      14        40      True
  Manhattan         True      14        14      True
  Euclidean         True      14        24      True
  2 x Manhattan     True      14        14      True

OPEN: BFS (optimal) length = 24, BFS expanded = 136
  heuristic        found  length  expanded  optimal?
  h = 0             True      24       136      True
  Manhattan         True      24        24      True
  Euclidean         True      24       106      True
  2 x Manhattan     True      24        24      True

TRAP: BFS (optimal) length = 18, BFS expand

In [8]:
# Visualise the TRAP map: optimal Manhattan path vs the inflated (2 x Manhattan) path
g_, s_, t_ = parse_map(TRAP)
print("Manhattan (admissible):")
print(draw(g_, astar(g_, s_, t_, manhattan)["path"]))
print("\n2 x Manhattan (inadmissible):")
print(draw(g_, astar(g_, s_, t_, double_manhattan)["path"]))

Manhattan (admissible):
#############
#S#.##......#
#*.#..#****.#
#*******##*.#
#....#..##*.#
#..#....#.*##
#...##..##**#
##.#......#G#
#############

2 x Manhattan (inadmissible):
#############
#S#.##......#
#*.#..#****.#
#*.....*##*.#
#****#.*##*.#
#..#****#.*##
#...##..##**#
##.#......#G#
#############


**Observations**

1. **$h = 0$:** A* becomes uniform-cost search, which with unit costs behaves like BFS. Always optimal, but expands the most states (same counts as BFS on every map).
2. **Euclidean:** still admissible ($\sqrt{\Delta r^2 + \Delta c^2} \le |\Delta r| + |\Delta c|$), so paths stay optimal. But it is a *smaller* (less informed) estimate than Manhattan, so it expands more states than Manhattan (e.g. on `OPEN`).
3. **2 × Manhattan:** no longer admissible – it can overestimate. It searches greedily toward `G`. On `ALT`/`OPEN` it happens to stay optimal, but on `TRAP` it returns a **longer, non-optimal path** (22 vs the optimal 18) – and here it did not even save work (43 expansions vs 42 for Manhattan). Inflating $h$ gives up the optimality guarantee; any speed-up is map-dependent.
4. On the lab's own warehouse map **all four heuristics give identical results** (length 40, 63 expanded), because the single long corridor forces every variant to explore essentially the whole map.

**Answer to the "Think About It":** the closer $h$ is to $h^*$ from below, the fewer nodes A* expands while staying optimal. $h = 0$ is "too pessimistic" and wastes work. Once $h$ overestimates ($h > h^*$ somewhere), A* can be lured onto a route that looks good and commit to it before discovering the cheaper one, so optimality is lost even though it gets faster.

### Task 7: Evaluate the LLM-generated agent

1. **Correct immediately:** grid parsing, the successor function, the heap-based frontier, path reconstruction, and BFS.
2. **Bugs / design problems found:**
   - The first version used heap entries `(f, g, state)`. On ties in `f` this prefers the **smallest g**, i.e. nodes nearest the *start*, which made A* expand far more than needed on the open map (127 states vs 24 after the fix). Changed to `(f, h, counter, g, state)`.
   - Comparing raw tuples containing states can fail if states ever stop being comparable; the `counter` tie-breaker removes that risk.
   - The expansion counter must not count stale heap entries that are skipped; I checked that it is incremented only after the `closed` check.
3. **How they were found:** by running A* against BFS on the open map – the expanded counts were suspiciously close to BFS even though Manhattan distance should be very informative in an open room.
4. **Unfamiliar terminology:** "lazy deletion" (leaving outdated heap entries and skipping them on pop) – looked up and understood before keeping it.
5. **Modifications:** tie-breaking in the heap tuple; heuristic made a parameter; added `draw()` and the `assert`-based tests.
6. **Most useful tests:** comparing path length against BFS (an independent optimality oracle), and the no-solution map (checks termination).
7. **Trust without testing?** No. The original version *did* return correct paths – the tie-breaking problem only showed up as a performance measure, which you would never notice from "it prints a path".
8. **What I understood better about A\*:** that the goal test must happen on expansion, not generation; that the number of states expanded depends on tie-breaking and on the map structure, not just on the heuristic; and that a heuristic that ignores obstacles can be almost useless in maze-like maps.

**Designed by me:** problem formulation, state/frontier design, test cases and expected results. **Suggested by the LLM:** the code structure, lazy deletion, and the Manhattan admissibility argument. **Accepted:** the structure and BFS. **Changed:** tie-breaking, heuristic parameter, tests and visualisation. **Tested:** all four lab tests plus the heuristic comparison on four maps.

### Final reflection

**1. Why formulate the problem first?** The algorithm only works on what the formulation defines: what a state is, which moves are legal, what counts as the goal and what things cost. If those are vague, the code (whether written by me or an LLM) will silently make its own choices, e.g. allowing diagonal moves or testing the goal on generation. Writing the formulation first also gives the expected results needed for testing.

**2. In what sense is A\* "informed"?** It uses problem-specific knowledge – an estimate $h(n)$ of the remaining cost – in addition to the cost so far $g(n)$. It expands nodes in order of $f = g + h$, i.e. by estimated total solution cost through that node, rather than blindly by depth (BFS) or cost so far (UCS).

**3. Why does the heuristic matter?** It decides both correctness and efficiency. An admissible heuristic keeps A* optimal; a more accurate admissible heuristic means fewer expansions. An inadmissible one can make A* greedier but return worse paths (seen on `TRAP`), and a heuristic that ignores the map's structure can give no benefit at all (seen on the lab warehouse).

**4. What did the LLM contribute?** A clean first implementation quickly, including standard details like lazy deletion and parent pointers, plus a correct explanation of why Manhattan distance is admissible. It saved typing and boilerplate; it did not decide what to test or interpret the results.

**5. What could go wrong without testing?** The code could return valid-looking but non-optimal paths (e.g. goal test on generation, or an inadmissible heuristic), loop forever on unsolvable maps, or be correct but much slower than it should be (the tie-breaking issue). None of these are visible from a single run that "prints a path".